In [3]:
from datasets import load_dataset

ds = load_dataset("saberzl/SID_Set", split="train", streaming=True)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

In [7]:
import os
from tqdm import tqdm
from datasets import load_dataset

# 1. Load the dataset (ensure 'ds' is defined)
print('📡 Loading streaming dataset...')
ds = load_dataset('saberzl/SID_Set', split='train', streaming=True)

# 2. Setup directories
base_path = 'data'
os.makedirs(f'{base_path}/real', exist_ok=True)
os.makedirs(f'{base_path}/fake', exist_ok=True)

real_count = 0
fake_count = 0
limit = 4000

print('📥 Starting data collection...')
# 3. Iterate and save
for item in tqdm(ds):
    label = item['label']

    # 0 = Real, 1 = Fake/Tampered in this dataset context
    if label == 0 and real_count < limit:
        item['image'].convert('RGB').save(f'{base_path}/real/real_{real_count}.jpg')
        real_count += 1
    elif label != 0 and fake_count < limit:
        item['image'].convert('RGB').save(f'{base_path}/fake/fake_{fake_count}.jpg')
        fake_count += 1

    if real_count >= limit and fake_count >= limit:
        break

print(f'✅ Done! Saved {real_count} real and {fake_count} fake images to {base_path}/')

📡 Loading streaming dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/249 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/34 [00:00<?, ?it/s]

📥 Starting data collection...


12062it [06:59, 28.75it/s]

✅ Done! Saved 4000 real and 4000 fake images to data/


In [12]:
#!/usr/bin/env python3
# SigLIP-2 (ViT) + SegFormer-style decoder for SID_Set

import os, math, argparse, random, csv, json
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import f1_score, accuracy_score
from transformers import SiglipVisionModel, SiglipImageProcessor

# --- Helper Classes and Functions ---

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class LinearProj(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__(); self.proj = nn.Linear(in_dim, out_dim)
    def forward(self, x): return self.proj(x)

class SegFormerStrongDecoder(nn.Module):
    def __init__(self, in_dims, embed_dim=256):
        super().__init__()
        self.projs = nn.ModuleList([LinearProj(d, embed_dim) for d in in_dims])
        self.fuse = nn.Sequential(nn.Conv2d(embed_dim*len(in_dims), embed_dim, 1), nn.GELU())
        self.head = nn.Conv2d(embed_dim, 1, 1)
    def forward(self, hidden_list, grid_hw, target_size=224):
        H, W = grid_hw
        feats = []
        for i, h in enumerate(hidden_list):
            x = self.projs[i](h).transpose(1, 2).reshape(h.shape[0], -1, H, W)
            feats.append(x)
        x = self.fuse(torch.cat(feats, dim=1))
        x = F.interpolate(x, size=(target_size, target_size), mode="bilinear", align_corners=False)
        return self.head(x)

class SigLIP2_MTL(nn.Module):
    def __init__(self, siglip_ckpt, seg_layers=(2, 6, 10, -1), embed_dim=256):
        super().__init__()
        self.encoder = SiglipVisionModel.from_pretrained(siglip_ckpt)
        hid = self.encoder.config.hidden_size
        self.cls_head = nn.Linear(hid, 1)
        self.seg_layers = seg_layers
        self.decoder = SegFormerStrongDecoder([hid]*len(seg_layers), embed_dim=embed_dim)
    def forward(self, pixel_values):
        out = self.encoder(pixel_values=pixel_values, output_hidden_states=True)
        pooled = out.pooler_output
        cls_logit = self.cls_head(pooled)
        hs = out.hidden_states
        feats = [hs[i if i>=0 else len(hs)-1] for i in self.seg_layers]
        N = feats[0].shape[1]; H = int(math.sqrt(N)); W = H
        seg_logits = self.decoder(feats, (H, W), target_size=pixel_values.shape[-1])
        return cls_logit, seg_logits

# --- Global Setup ---
args = type('Args', (), {
    'data_dir': './data',
    'siglip_ckpt': 'google/siglip2-base-patch16-224',
    'epochs': 3,
    'lr': 2e-5,
    'embed_dim': 256,
    'seed': 42
})()

set_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Load Model
model = SigLIP2_MTL(args.siglip_ckpt, embed_dim=args.embed_dim).to(device)
processor = SiglipImageProcessor.from_pretrained(args.siglip_ckpt)

# 2. Setup Data
from torchvision import transforms
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std)
])

if os.path.exists(args.data_dir):
    dataset = ImageFolder(args.data_dir, transform=transform)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)

    # 3. Optimizer & Loss
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr)
    criterion_cls = nn.BCEWithLogitsLoss()

    print(f"✨ Starting training on {device}...")
    for epoch in range(1, args.epochs + 1):
        model.train()
        total_loss = 0
        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}"):
            imgs, labels = imgs.to(device), labels.float().to(device).unsqueeze(1)
            optimizer.zero_grad()
            cls_logits, _ = model(imgs)
            loss = criterion_cls(cls_logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch} Loss: {total_loss/len(train_loader):.4f}")
else:
    print("❌ Data directory not found. Please run data collection.")

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SiglipVisionModel LOAD REPORT from: google/siglip2-base-patch16-224
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
text_model.head.bias                                         | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_pro

✨ Starting training on cuda...


Epoch 1: 100%|██████████| 800/800 [05:37<00:00,  2.37it/s]


Epoch 1 Loss: 0.4148


Epoch 2: 100%|██████████| 800/800 [05:36<00:00,  2.38it/s]


Epoch 2 Loss: 0.2485


Epoch 3: 100%|██████████| 800/800 [05:36<00:00,  2.38it/s]

Epoch 3 Loss: 0.1439


In [13]:
import torch
from sklearn.metrics import classification_report, f1_score, accuracy_score
from tqdm import tqdm

def evaluate_and_save(model, val_loader, device, save_path='siglip2_sid_model.pt'):
    model.eval()
    all_preds = []
    all_labels = []

    print("\n🔍 Starting Evaluation...")
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc="Evaluating"):
            imgs = imgs.to(device)
            cls_logits, _ = model(imgs)
            preds = (torch.sigmoid(cls_logits) > 0.5).cpu().numpy().astype(int)

            all_preds.extend(preds.flatten())
            all_labels.extend(labels.numpy().astype(int))

    # Calculate Metrics
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    print(f"\n📊 Final Results:")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print("\nDetailed Report:")
    print(classification_report(all_labels, all_preds, target_names=['Real', 'Fake']))

    # Save Model
    torch.save(model.state_dict(), save_path)
    print(f"\n💾 Model weights saved to: {save_path}")

# Run evaluation
if 'model' in globals() and 'val_loader' in globals():
    evaluate_and_save(model, val_loader, device)
else:
    print("❌ Training variables not found. Please ensure the training cell has finished executing.")


🔍 Starting Evaluation...


Evaluating: 100%|██████████| 200/200 [00:41<00:00,  4.82it/s]



📊 Final Results:
Accuracy: 0.8594
F1 Score: 0.8645

Detailed Report:
              precision    recall  f1-score   support

        Real       0.89      0.82      0.85       799
        Fake       0.83      0.90      0.86       801

    accuracy                           0.86      1600
   macro avg       0.86      0.86      0.86      1600
weighted avg       0.86      0.86      0.86      1600


💾 Model weights saved to: siglip2_sid_model.pt
